# Heat Transfer and Thermal Equilibrium — Particle Model

Three panels show a hot red material, a cooler blue thermometer placed in contact, and finally thermal equilibrium.


In [ ]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation

from matplotlib.patches import Rectangle

from matplotlib.transforms import Affine2D

from IPython.display import HTML


In [ ]:
def animate_heat_demo(delta_hot=0.24, delta_cold=0.05, delta_equilibrium=0.14, frames=140, interval=60, seed=4):

    rng = np.random.default_rng(seed)



    def make_grid(nx, ny, x0, x1, y0, y1):

        xs = np.linspace(x0, x1, nx)

        ys = np.linspace(y0, y1, ny)

        X, Y = np.meshgrid(xs, ys)

        return X.ravel(), Y.ravel()



    # 400 red particles: 20 x 20

    red_x0, red_y0 = make_grid(20, 20, -2.2, 2.2, -2.2, 2.2)



    # 120 blue thermometer particles: 6 x 20

    bx, by = make_grid(6, 20, -0.35, 0.35, -2.0, 2.0)

    angle_deg = -45

    theta = np.deg2rad(angle_deg)

    c, s = np.cos(theta), np.sin(theta)

    thermometer_cx = 0.7

    thermometer_cy = 0.9

    blue_x0 = c*bx - s*by + thermometer_cx

    blue_y0 = s*bx + c*by + thermometer_cy



    def vibration_params(n):

        return (rng.uniform(0,2*np.pi,n), rng.uniform(0,2*np.pi,n), rng.uniform(0,2*np.pi,n), rng.uniform(0.8,1.35,n))



    red_theta, red_phase, red_phase2, red_freq = vibration_params(len(red_x0))

    blue_theta, blue_phase, blue_phase2, blue_freq = vibration_params(len(blue_x0))



    fig, axes = plt.subplots(1, 3, figsize=(15,5))

    titles = ['1. Hot material', '2. Cooler thermometer in contact', '3. Thermal equilibrium']

    for ax, title in zip(axes, titles):

        ax.set_xlim(-3.6,3.6); ax.set_ylim(-3.6,3.6); ax.set_aspect('equal')

        ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)

    color1 = "blue"

    color2 = "pink"

    def add_thermometer_box(ax, color1):

        width = 0.95

    

        # Blue particles run locally from y = -2 to y = +2.

        # Start tube just below the lowest blue particles.

        bottom = -2.15

    

        # Deliberately long so the top disappears off screen

        height = 10.0

    

        # Outer thermometer tube

        rect = Rectangle(

            (-width/2, bottom),

            width,

            height,

            facecolor='white',

            linewidth=2,

            edgecolor='black',

            zorder=4

        )

    

        # Blue liquid strip:

        # starts near the top of the blue particle region and continues upward

        strip_width = 0.32

        strip_bottom = 1.8

    

        strip = Rectangle(

            (-strip_width/2, strip_bottom),

            strip_width,

            height,

            facecolor=color1,

            edgecolor='none',

            zorder=5

        )

    

        trans = (

            Affine2D()

            .rotate_deg(angle_deg)

            .translate(thermometer_cx, thermometer_cy)

            + ax.transData

        )

    

        rect.set_transform(trans)

        strip.set_transform(trans)

    

        ax.add_patch(strip)

        ax.add_patch(rect)

    

        return rect, strip

        

    red1 = axes[0].scatter(red_x0, red_y0, s=20, color=color2, zorder=2)

    red2 = axes[1].scatter(red_x0, red_y0, s=20, color=color2, zorder=2)

    red3 = axes[2].scatter(red_x0, red_y0, s=20, color=color2, zorder=2)

    

    # White thermometer body goes above red beads

    add_thermometer_box(axes[1], color1)

    add_thermometer_box(axes[2], color1)

    

    # Blue beads go above thermometer body

    blue2 = axes[1].scatter(blue_x0, blue_y0, s=20, color=color1, zorder=5)

    blue3 = axes[2].scatter(blue_x0, blue_y0, s=20, color=color1, zorder=5)



    def vibrating_positions(x0,y0,amp,th,ph1,ph2,freq,t):

        u = amp*np.sin(freq*t + ph1)

        v = 0.35*amp*np.sin(1.37*freq*t + ph2)

        dx = u*np.cos(th) - v*np.sin(th)

        dy = u*np.sin(th) + v*np.cos(th)

        return np.column_stack((x0+dx, y0+dy))



    def update(frame):

        t = frame*0.16

        p_red_hot = vibrating_positions(red_x0,red_y0,delta_hot,red_theta,red_phase,red_phase2,red_freq,t)

        red1.set_offsets(p_red_hot); red2.set_offsets(p_red_hot)

        p_blue_cold = vibrating_positions(blue_x0,blue_y0,delta_cold,blue_theta,blue_phase,blue_phase2,blue_freq,t)

        blue2.set_offsets(p_blue_cold)

        p_red_eq = vibrating_positions(red_x0,red_y0,delta_equilibrium,red_theta,red_phase,red_phase2,red_freq,t)

        p_blue_eq = vibrating_positions(blue_x0,blue_y0,delta_equilibrium,blue_theta,blue_phase,blue_phase2,blue_freq,t)

        red3.set_offsets(p_red_eq); blue3.set_offsets(p_blue_eq)

        return red1, red2, blue2, red3, blue3



    anim = FuncAnimation(fig, update, frames=frames, interval=interval, blit=False)

    plt.close(fig)

    return HTML(anim.to_jshtml())



animate_heat_demo()


### Interpretation

- Red material starts with the larger vibration amplitude.
- Blue thermometer starts with the smaller vibration amplitude.
- In the final panel both use the same vibration amplitude.
